In [ ]:
# Installation for library

import os
import json
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from scipy import stats as sp_stats

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D, Flatten,
    Dropout, LSTM, Bidirectional, concatenate, Layer
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
print(f"TensorFlow : {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# CELL 1: MOUNT GOOGLE DRIVE & SET PATHS

from google.colab import drive
import os
drive.mount('/content/drive')

# EXACT path from your screenshot (case-sensitive!)
WESAD_PATH = "/content/drive/MyDrive/WESAD/wesad"

# All results (figures + JSON) will be saved here automatically
RESULTS_DIR = "/content/drive/MyDrive/HABC_Results_WESAD"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify the dataset exists (Now correctly looking for .pkl FILES, not folders)
subject_files = sorted([
    f for f in os.listdir(WESAD_PATH)
    if f.endswith('.pkl') and f.lower().startswith('s')
])

print(f"Reading from path: {WESAD_PATH}")
print(f"Found {len(subject_files)} subject files: {subject_files}")
print(f"Results will be saved to: {RESULTS_DIR}")

In [ ]:
# CELL 2: ATTENTION LAYER + ALL MODEL BUILDERS (same architecture as UTD)

# ─────────────── Custom Attention Layer ───────────────

class AttentionLayer(Layer):
    """Temporal attention: learns which time steps matter most."""

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight", shape=(input_shape[-1], 1), initializer="normal"
        )
        self.b = self.add_weight(
            name="att_bias", shape=(input_shape[1], 1), initializer="zeros"
        )
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at = K.expand_dims(at, axis=-1)
        output = x * at
        return K.sum(output, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        return super(AttentionLayer, self).get_config()


# ─────────────── Model Builders ───────────────

def build_habc_model(num_classes, time_steps, feature_dim, lr=0.001):
    """FULL HABC: CNN + BiLSTM + Attention (the proposed model)."""
    # CNN branch
    cnn_inp = Input(shape=(1, time_steps, feature_dim), name='cnn_input')
    c = Conv2D(6, (3, 3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1, 2), strides=(1, 2), padding='same')(c)
    c = Flatten()(c)

    # BiLSTM + Attention branch
    lstm_inp = Input(shape=(time_steps, feature_dim), name='lstm_input')
    l = Bidirectional(LSTM(64, return_sequences=True))(lstm_inp)
    l = AttentionLayer()(l)

    # Fusion
    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        metrics=['accuracy']
    )
    return model


def build_cnn_only(num_classes, time_steps, feature_dim, lr=0.001):
    """Ablation variant: CNN branch only."""
    inp = Input(shape=(1, time_steps, feature_dim))
    x = Conv2D(6, (3, 3), padding='same', activation='relu')(inp)
    x = MaxPooling2D(pool_size=(1, 2), strides=(1, 2), padding='same')(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


def build_bilstm_only(num_classes, time_steps, feature_dim, lr=0.001):
    """Ablation variant: BiLSTM only (no attention, no CNN)."""
    inp = Input(shape=(time_steps, feature_dim))
    x = Bidirectional(LSTM(64, return_sequences=False))(inp)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


def build_cnn_bilstm_no_attention(num_classes, time_steps, feature_dim, lr=0.001):
    """Ablation variant: CNN + BiLSTM but NO attention mechanism."""
    cnn_inp = Input(shape=(1, time_steps, feature_dim))
    c = Conv2D(6, (3, 3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1, 2), strides=(1, 2), padding='same')(c)
    c = Flatten()(c)

    lstm_inp = Input(shape=(time_steps, feature_dim))
    l = Bidirectional(LSTM(64, return_sequences=False))(lstm_inp)

    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m


def build_bilstm_attention_only(num_classes, time_steps, feature_dim, lr=0.001):
    """Ablation variant: BiLSTM + Attention only (no CNN)."""
    inp = Input(shape=(time_steps, feature_dim))
    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = AttentionLayer()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr), metrics=['accuracy'])
    return m

In [ ]:
# =============================================================================
# CELL 3: LOAD & PREPROCESS WESAD DATA
# =============================================================================

import os
import pickle
import numpy as np
import scipy.stats as sp_stats
from collections import Counter
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

def load_wesad(wesad_path, window_seconds=10, overlap_ratio=0.5, downsample_factor=10):
    """
    Load WESAD chest sensor data from per-subject .pkl files.
    """
    CHEST_KEYS = ['ACC', 'ECG', 'EDA', 'EMG', 'Resp', 'Temp']
    VALID_LABELS = [1, 2, 3]
    RAW_RATE = 700  # Hz

    window_size_raw = RAW_RATE * window_seconds
    step_raw = int(window_size_raw * (1 - overlap_ratio))
    window_size_ds = window_size_raw // downsample_factor   # time_steps per window

    X_all, y_all, subj_all = [], [], []

    # FIX: Loop directly over .pkl files instead of folders
    for fname in sorted(os.listdir(wesad_path)):
        if not fname.endswith('.pkl') or not fname.lower().startswith('s'):
            continue

        pkl_file = os.path.join(wesad_path, fname)

        # Extract Subject ID (e.g., 'S2.pkl' -> 2)
        try:
            sid = int(fname.lower().replace('s', '').replace('.pkl', ''))
        except ValueError:
            continue

        print(f"  Loading {fname} ...", end=" ")

        with open(pkl_file, 'rb') as f:
            data = pickle.load(f, encoding='latin1')

        labels = data['label'].flatten()                      # (total_samples,)
        chest  = data['signal']['chest']

        # Stack all chest signals → (total_samples, 8)
        sig_parts = []
        for key in CHEST_KEYS:
            sig = chest[key]
            if sig.ndim == 1:
                sig = sig.reshape(-1, 1)
            sig_parts.append(sig)
        all_signals = np.hstack(sig_parts).astype('float32')  # (total_samples, 8)

        n_windows = 0
        for start in range(0, len(all_signals) - window_size_raw + 1, step_raw):
            end = start + window_size_raw
            win_labels = labels[start:end]

            # At least 80 % of the window must have a valid label
            valid_mask = np.isin(win_labels, VALID_LABELS)
            if valid_mask.mean() < 0.8:
                continue

            # Window label = statistical mode of valid samples
            valid_lbls = win_labels[valid_mask]
            mode_result = sp_stats.mode(valid_lbls, keepdims=False)

            # Handling older/newer scipy versions
            win_label = int(mode_result.mode[0] if isinstance(mode_result.mode, np.ndarray) else mode_result.mode)

            if win_label not in VALID_LABELS:
                continue

            # Downsample: take every Nth sample
            win_data = all_signals[start:end:downsample_factor]   # (window_size_ds, 8)
            if win_data.shape[0] != window_size_ds:
                continue

            X_all.append(win_data)
            y_all.append(win_label - 1)   # → 0=Baseline, 1=Stress, 2=Amusement
            subj_all.append(sid)
            n_windows += 1

        print(f"→ {n_windows} windows")

    # Safety Check to prevent the ValueError
    if len(X_all) == 0:
        raise ValueError(f"No data was extracted! Please check if {wesad_path} contains the .pkl files.")

    X_all = np.array(X_all, dtype='float32')
    y_all = np.array(y_all)
    subj_all = np.array(subj_all)

    # Z-score normalization per feature (across all windows)
    N, T, F = X_all.shape
    X_flat = X_all.reshape(-1, F)
    scaler = StandardScaler()
    X_flat = scaler.fit_transform(X_flat)
    X_all = X_flat.reshape(N, T, F).astype('float32')

    return X_all, y_all, subj_all


# ──────────── Actually load the data ────────────

print("=" * 60)
print("Loading WESAD dataset …")
print("=" * 60)

raw_X, raw_y, raw_subjects = load_wesad(WESAD_PATH,
                                         window_seconds=10,
                                         overlap_ratio=0.5,
                                         downsample_factor=10)

TIME_STEPS  = raw_X.shape[1]        # e.g. 700
FEATURE_DIM = raw_X.shape[2]        # 8
NUM_CLASSES = len(np.unique(raw_y))  # 3

CLASS_NAMES = {0: 'Baseline', 1: 'Stress', 2: 'Amusement'}
CLASS_LIST  = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]

print(f"\n{'=' * 60}")
print(f"  WESAD Data Summary")
print(f"{'=' * 60}")
print(f"  Total windows : {len(raw_X)}")
print(f"  Time steps    : {TIME_STEPS}")
print(f"  Features      : {FEATURE_DIM}  (ACC×3, ECG, EDA, EMG, Resp, Temp)")
print(f"  Classes       : {NUM_CLASSES} → {CLASS_NAMES}")
print(f"  Subjects      : {sorted(np.unique(raw_subjects))} ({len(np.unique(raw_subjects))} total)")
print(f"  Class counts  : {dict(Counter(raw_y))}")
print(f"  X shape       : {raw_X.shape}")

# Prepare two-input formats for HABC
X_cnn_all  = raw_X.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_all = raw_X.copy()
y_encoded  = to_categorical(raw_y, num_classes=NUM_CLASSES)

print(f"\n  CNN  input : {X_cnn_all.shape}")
print(f"  LSTM input : {X_lstm_all.shape}")
print(f"  Labels     : {y_encoded.shape}")

In [ ]:
# CELL 4: QUICK SANITY CHECK — Train HABC on full data

print("\n" + "=" * 60)
print("SANITY CHECK: Training HABC on full WESAD data")
print("=" * 60)

K.clear_session()
model_sanity = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
model_sanity.summary()

es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
history = model_sanity.fit(
    [X_cnn_all, X_lstm_all], y_encoded,
    epochs=100, batch_size=64, callbacks=[es], verbose=1
)

score = model_sanity.evaluate([X_cnn_all, X_lstm_all], y_encoded, verbose=0)
print(f"\nTraining-set accuracy (sanity check): {score[1]*100:.2f}%")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], linewidth=2)
ax1.set(title='Training Loss', xlabel='Epoch', ylabel='Loss')
ax1.grid(True, alpha=0.3)
ax2.plot(history.history['accuracy'], linewidth=2, color='green')
ax2.set(title='Training Accuracy', xlabel='Epoch', ylabel='Accuracy')
ax2.grid(True, alpha=0.3)
plt.suptitle('HABC Training on WESAD (sanity check)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/sanity_training_curves.png", dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# CELL 5: EXPERIMENT 1 — CROSS-SUBJECT GENERALIZATION

print("\n" + "=" * 60)
print("EXPERIMENT 1: Cross-Subject Generalization — WESAD")
print("=" * 60)

# ─────────── Helper ───────────
def train_and_evaluate_habc(X_tr, X_te, y_tr_raw, y_te_raw,
                            num_classes, ts, fd, epochs=100, bs=64):
    """Build, train, evaluate a fresh HABC and return metrics."""
    y_tr = to_categorical(y_tr_raw, num_classes=num_classes)
    y_te = to_categorical(y_te_raw, num_classes=num_classes)

    X_cnn_tr = X_tr.reshape(-1, 1, ts, fd)
    X_lstm_tr = X_tr.copy()
    X_cnn_te = X_te.reshape(-1, 1, ts, fd)
    X_lstm_te = X_te.copy()

    K.clear_session()
    model = build_habc_model(num_classes, ts, fd)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit([X_cnn_tr, X_lstm_tr], y_tr,
              epochs=epochs, batch_size=bs, verbose=0, callbacks=[es])

    y_pred = np.argmax(model.predict([X_cnn_te, X_lstm_te], verbose=0), axis=1)
    y_true = y_te_raw

    acc   = accuracy_score(y_true, y_pred)
    mf1   = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    wf1   = f1_score(y_true, y_pred, average='weighted',  zero_division=0)
    return acc, mf1, wf1, y_pred, y_true


# ─────────── 1A: Leave-One-Subject-Out (LOSO) ───────────
print("\n--- 1A: Leave-One-Subject-Out (LOSO) ---")

logo = LeaveOneGroupOut()
n_folds = len(np.unique(raw_subjects))
print(f"Running {n_folds} LOSO folds (one per subject)…\n")

loso_results = []

for fold, (tr_idx, te_idx) in enumerate(logo.split(raw_X, raw_y, raw_subjects)):
    te_subj = np.unique(raw_subjects[te_idx])[0]

    acc, mf1, wf1, _, _ = train_and_evaluate_habc(
        raw_X[tr_idx], raw_X[te_idx],
        raw_y[tr_idx], raw_y[te_idx],
        NUM_CLASSES, TIME_STEPS, FEATURE_DIM
    )

    loso_results.append({
        'subject': int(te_subj), 'accuracy': acc,
        'macro_f1': mf1, 'weighted_f1': wf1,
        'n_test': len(te_idx)
    })
    print(f"  Fold {fold+1:2d}/{n_folds} | Subject S{te_subj:2d} | "
          f"Acc={acc*100:6.2f}%  Macro-F1={mf1*100:6.2f}%  "
          f"Weighted-F1={wf1*100:6.2f}%  (n={len(te_idx)})")

loso_accs = [r['accuracy']   for r in loso_results]
loso_mf1  = [r['macro_f1']   for r in loso_results]
loso_wf1  = [r['weighted_f1'] for r in loso_results]

print(f"\n  LOSO RESULTS:")
print(f"  Accuracy    : {np.mean(loso_accs)*100:.2f}% ± {np.std(loso_accs)*100:.2f}%")
print(f"  Macro-F1    : {np.mean(loso_mf1)*100:.2f}% ± {np.std(loso_mf1)*100:.2f}%")
print(f"  Weighted-F1 : {np.mean(loso_wf1)*100:.2f}% ± {np.std(loso_wf1)*100:.2f}%")


# ─────────── 1B: Cross-Subject Split (70 % train / 30 % test) ───────────
print("\n--- 1B: Cross-Subject Split (70/30 by subject, 5 random splits) ---")

gss = GroupShuffleSplit(n_splits=5, test_size=0.3, random_state=42)
cs_results = []

for split_i, (tr_idx, te_idx) in enumerate(gss.split(raw_X, raw_y, raw_subjects)):
    te_subs = sorted(np.unique(raw_subjects[te_idx]))
    acc, mf1, wf1, _, _ = train_and_evaluate_habc(
        raw_X[tr_idx], raw_X[te_idx],
        raw_y[tr_idx], raw_y[te_idx],
        NUM_CLASSES, TIME_STEPS, FEATURE_DIM
    )
    cs_results.append({'split': split_i+1, 'accuracy': acc,
                       'macro_f1': mf1, 'weighted_f1': wf1,
                       'test_subjects': [int(s) for s in te_subs]})
    print(f"  Split {split_i+1}/5 | Test subjects: {te_subs} | "
          f"Acc={acc*100:.2f}%  Macro-F1={mf1*100:.2f}%")

cs_accs = [r['accuracy']  for r in cs_results]
cs_mf1  = [r['macro_f1']  for r in cs_results]
cs_wf1  = [r['weighted_f1'] for r in cs_results]

print(f"\n  CROSS-SUBJECT RESULTS:")
print(f"  Accuracy    : {np.mean(cs_accs)*100:.2f}% ± {np.std(cs_accs)*100:.2f}%")
print(f"  Macro-F1    : {np.mean(cs_mf1)*100:.2f}% ± {np.std(cs_mf1)*100:.2f}%")
print(f"  Weighted-F1 : {np.mean(cs_wf1)*100:.2f}% ± {np.std(cs_wf1)*100:.2f}%")

# Save Experiment 1
exp1 = {
    "experiment": "Exp1_Cross_Subject",
    "dataset": "WESAD",
    "LOSO": {
        "mean_acc": float(np.mean(loso_accs)), "std_acc": float(np.std(loso_accs)),
        "mean_macro_f1": float(np.mean(loso_mf1)), "std_macro_f1": float(np.std(loso_mf1)),
        "mean_weighted_f1": float(np.mean(loso_wf1)),
        "per_subject": loso_results
    },
    "Cross_Subject": {
        "mean_acc": float(np.mean(cs_accs)), "std_acc": float(np.std(cs_accs)),
        "mean_macro_f1": float(np.mean(cs_mf1)),
        "mean_weighted_f1": float(np.mean(cs_wf1)),
        "per_split": cs_results
    }
}
with open(f"{RESULTS_DIR}/exp1_cross_subject.json", "w") as f:
    json.dump(exp1, f, indent=2, default=str)
print(f"\nSaved → {RESULTS_DIR}/exp1_cross_subject.json")



In [ ]:
# CELL 6: EXPERIMENT 2 — ABLATION STUDY
print("\n" + "=" * 60)
print("EXPERIMENT 2: Ablation Study — WESAD")
print("=" * 60)

# Fixed cross-subject split for fair comparison across all variants
gss_ab = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_ab, te_ab = next(gss_ab.split(raw_X, raw_y, raw_subjects))

X_tr_ab, X_te_ab = raw_X[tr_ab], raw_X[te_ab]
y_tr_ab_raw, y_te_ab_raw = raw_y[tr_ab], raw_y[te_ab]
y_tr_ab = to_categorical(y_tr_ab_raw, num_classes=NUM_CLASSES)
y_te_ab = to_categorical(y_te_ab_raw, num_classes=NUM_CLASSES)

X_cnn_tr_ab  = X_tr_ab.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_tr_ab = X_tr_ab.copy()
X_cnn_te_ab  = X_te_ab.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_te_ab = X_te_ab.copy()

ablation_config = {
    "CNN only": {
        "build": build_cnn_only,
        "train_in": [X_cnn_tr_ab],
        "test_in":  [X_cnn_te_ab],
    },
    "BiLSTM only": {
        "build": build_bilstm_only,
        "train_in": [X_lstm_tr_ab],
        "test_in":  [X_lstm_te_ab],
    },
    "BiLSTM + Attention": {
        "build": build_bilstm_attention_only,
        "train_in": [X_lstm_tr_ab],
        "test_in":  [X_lstm_te_ab],
    },
    "CNN + BiLSTM (no Attn)": {
        "build": build_cnn_bilstm_no_attention,
        "train_in": [X_cnn_tr_ab, X_lstm_tr_ab],
        "test_in":  [X_cnn_te_ab, X_lstm_te_ab],
    },
    "HABC (Full)": {
        "build": build_habc_model,
        "train_in": [X_cnn_tr_ab, X_lstm_tr_ab],
        "test_in":  [X_cnn_te_ab, X_lstm_te_ab],
    },
}

ablation_results = {}

for name, cfg in ablation_config.items():
    print(f"\n  Training: {name} …")
    K.clear_session()
    model = cfg["build"](NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit(cfg["train_in"], y_tr_ab,
              epochs=100, batch_size=64, verbose=0, callbacks=[es])

    y_pred = np.argmax(model.predict(cfg["test_in"], verbose=0), axis=1)
    y_true = y_te_ab_raw

    acc  = accuracy_score(y_true, y_pred)
    mf1  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)

    ablation_results[name] = {
        'accuracy': acc, 'macro_f1': mf1, 'precision': prec, 'recall': rec
    }
    print(f"    → Acc={acc*100:.2f}%  Macro-F1={mf1*100:.2f}%  "
          f"Prec={prec*100:.2f}%  Rec={rec*100:.2f}%")

# Print table
print(f"\n{'='*80}")
print(f"  {'Model Variant':<30} {'Accuracy':>10} {'Macro-F1':>10} {'Precision':>10} {'Recall':>10}")
print(f"{'='*80}")
for name, r in ablation_results.items():
    tag = " ◀ proposed" if "HABC" in name else ""
    print(f"  {name:<30} {r['accuracy']*100:>9.2f}% {r['macro_f1']*100:>9.2f}% "
          f"{r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}%{tag}")
print(f"{'='*80}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
names = list(ablation_results.keys())
accs = [ablation_results[n]['accuracy']*100 for n in names]
f1s  = [ablation_results[n]['macro_f1']*100  for n in names]
x = np.arange(len(names))
w = 0.35
b1 = ax.bar(x - w/2, accs, w, label='Accuracy (%)', color='#2196F3')
b2 = ax.bar(x + w/2, f1s,  w, label='Macro-F1 (%)',  color='#FF9800')
ax.set_ylabel('Score (%)')
ax.set_title('Ablation Study — WESAD', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)
for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{b.get_height():.1f}', ha='center', fontsize=8)
for b in b2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{b.get_height():.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp2_ablation.png", dpi=300, bbox_inches='tight')
plt.show()

with open(f"{RESULTS_DIR}/exp2_ablation.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
print(f"Saved → {RESULTS_DIR}/exp2_ablation.json + exp2_ablation.png")


In [ ]:
# CELL 7: EXPERIMENT 3 — ROBUSTNESS TO NOISE & MISSING DATA

print("\n" + "=" * 60)
print("EXPERIMENT 3: Robustness to Noise & Missing Data — WESAD")
print("=" * 60)

# Train HABC on clean training data (same split as ablation)
K.clear_session()
model_robust = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_robust.fit([X_cnn_tr_ab, X_lstm_tr_ab], y_tr_ab,
                 epochs=100, batch_size=64, verbose=0, callbacks=[es])

# Baseline on clean test data
y_pred_clean = np.argmax(model_robust.predict([X_cnn_te_ab, X_lstm_te_ab], verbose=0), axis=1)
clean_acc = accuracy_score(y_te_ab_raw, y_pred_clean)
print(f"\n  Clean test accuracy: {clean_acc*100:.2f}%")


# ── 3A: Gaussian Noise ──
print("\n--- 3A: Gaussian Noise (SNR in dB) ---")

def add_gaussian_noise(X, snr_db, seed=42):
    rng = np.random.RandomState(seed)
    sig_power = np.mean(X ** 2)
    if sig_power == 0:
        return X.copy()
    noise_power = sig_power / (10 ** (snr_db / 10))
    return X + rng.normal(0, np.sqrt(noise_power), X.shape).astype('float32')

snr_levels = [30, 20, 10, 5, 0]
noise_results = {'clean': float(clean_acc)}

for snr in snr_levels:
    X_noisy = add_gaussian_noise(X_te_ab, snr)
    X_c = X_noisy.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
    X_l = X_noisy.copy()
    y_p = np.argmax(model_robust.predict([X_c, X_l], verbose=0), axis=1)
    acc = accuracy_score(y_te_ab_raw, y_p)
    noise_results[f'snr_{snr}'] = float(acc)
    print(f"  SNR = {snr:3d} dB  →  Accuracy = {acc*100:.2f}%")


# ── 3B: Missing Data ──
print("\n--- 3B: Missing Data Simulation ---")

def simulate_missing(X, pct, seed=42):
    rng = np.random.RandomState(seed)
    Xm = X.copy()
    mask = rng.random(X.shape) < pct
    Xm[mask] = 0.0
    return Xm

miss_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]
miss_results = {}

for pct in miss_levels:
    Xm = simulate_missing(X_te_ab, pct)
    X_c = Xm.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
    X_l = Xm.copy()
    y_p = np.argmax(model_robust.predict([X_c, X_l], verbose=0), axis=1)
    acc = accuracy_score(y_te_ab_raw, y_p)
    miss_results[f'{pct*100:.0f}%'] = float(acc)
    print(f"  Missing = {pct*100:5.1f}%  →  Accuracy = {acc*100:.2f}%")


# ── 3C: Plots ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Noise
snr_x = snr_levels
snr_y = [noise_results[f'snr_{s}']*100 for s in snr_x]
ax1.plot(snr_x, snr_y, 'bo-', linewidth=2, markersize=8)
ax1.axhline(clean_acc*100, color='g', ls='--', alpha=.7, label=f'Clean ({clean_acc*100:.1f}%)')
ax1.set(xlabel='SNR (dB)', ylabel='Accuracy (%)',
        title='Robustness to Gaussian Noise')
ax1.invert_xaxis()
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 105)

# Missing
miss_x = [0, 5, 10, 15, 20, 30]
miss_y = [miss_results[f'{p}%']*100 for p in miss_x]
ax2.plot(miss_x, miss_y, 'rs-', linewidth=2, markersize=8)
ax2.set(xlabel='Missing Data (%)', ylabel='Accuracy (%)',
        title='Robustness to Missing Data')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 105)

plt.suptitle('Robustness Analysis — WESAD', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp3_robustness.png", dpi=300, bbox_inches='tight')
plt.show()

exp3 = {
    "experiment": "Exp3_Robustness", "dataset": "WESAD",
    "clean_accuracy": float(clean_acc),
    "noise": noise_results, "missing": miss_results
}
with open(f"{RESULTS_DIR}/exp3_robustness.json", "w") as f:
    json.dump(exp3, f, indent=2)
print(f"Saved → {RESULTS_DIR}/exp3_robustness.json + exp3_robustness.png")


In [ ]:
# CELL 8: EXPERIMENT 4 — STATISTICAL SIGNIFICANCE TESTING

print("\n" + "=" * 60)
print("EXPERIMENT 4: Statistical Significance — WESAD")
print("=" * 60)
print("Running LOSO for CNN-only and BiLSTM-only baselines to get per-fold results …\n")

# We already have HABC's per-fold accuracies from Exp 1
baseline_configs = {
    "CNN only": {"build": build_cnn_only, "mode": "cnn"},
    "BiLSTM only": {"build": build_bilstm_only, "mode": "lstm"},
}

sig_folds = {"HABC": loso_accs}  # from Experiment 1

for bl_name, bl_cfg in baseline_configs.items():
    print(f"  Running LOSO for: {bl_name}")
    bl_accs = []

    for fold, (tr_idx, te_idx) in enumerate(logo.split(raw_X, raw_y, raw_subjects)):
        X_tr, X_te = raw_X[tr_idx], raw_X[te_idx]
        y_tr = to_categorical(raw_y[tr_idx], num_classes=NUM_CLASSES)
        y_te_raw_fold = raw_y[te_idx]

        if bl_cfg["mode"] == "cnn":
            tr_in = [X_tr.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)]
            te_in = [X_te.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)]
        else:
            tr_in = [X_tr.copy()]
            te_in = [X_te.copy()]

        K.clear_session()
        m = bl_cfg["build"](NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
        es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
        m.fit(tr_in, y_tr, epochs=100, batch_size=64, verbose=0, callbacks=[es])

        y_p = np.argmax(m.predict(te_in, verbose=0), axis=1)
        bl_accs.append(accuracy_score(y_te_raw_fold, y_p))

        if (fold+1) % 5 == 0:
            print(f"    … fold {fold+1}/{n_folds} done")

    sig_folds[bl_name] = bl_accs
    print(f"  {bl_name} LOSO mean: {np.mean(bl_accs)*100:.2f}% ± {np.std(bl_accs)*100:.2f}%\n")


# Run paired statistical tests
print(f"\n{'='*75}")
print(f"  {'Comparison':<30} {'HABC':>8} {'Baseline':>10} {'t-stat':>8} {'p(t)':>10} {'W-stat':>8} {'p(W)':>10}")
print(f"{'='*75}")

habc_arr = np.array(sig_folds["HABC"])
sig_results = {}

for bl_name in baseline_configs:
    bl_arr = np.array(sig_folds[bl_name])

    t_stat, p_t = sp_stats.ttest_rel(habc_arr, bl_arr)
    try:
        w_stat, p_w = sp_stats.wilcoxon(habc_arr, bl_arr)
    except ValueError:
        w_stat, p_w = float('nan'), float('nan')

    sig_results[bl_name] = {
        'habc_mean': float(np.mean(habc_arr)),
        'baseline_mean': float(np.mean(bl_arr)),
        't_stat': float(t_stat), 'p_ttest': float(p_t),
        'w_stat': float(w_stat), 'p_wilcoxon': float(p_w),
        'significant_005': bool(p_t < 0.05)
    }

    pt_s = f"{p_t:.2e}" if p_t < 0.001 else f"{p_t:.6f}"
    pw_s = f"{p_w:.2e}" if (not np.isnan(p_w) and p_w < 0.001) else f"{p_w:.6f}"

    print(f"  HABC vs {bl_name:<20} {np.mean(habc_arr)*100:>7.2f}% "
          f"{np.mean(bl_arr)*100:>9.2f}% {t_stat:>8.3f} {pt_s:>10} "
          f"{w_stat:>8.1f} {pw_s:>10}")

print(f"{'='*75}")

exp4 = {
    "experiment": "Exp4_Significance", "dataset": "WESAD",
    "fold_accs": {k: [float(v) for v in vs] for k, vs in sig_folds.items()},
    "tests": sig_results
}
with open(f"{RESULTS_DIR}/exp4_significance.json", "w") as f:
    json.dump(exp4, f, indent=2)
print(f"\nSaved → {RESULTS_DIR}/exp4_significance.json")



In [ ]:
# CELL 9: EXPERIMENT 8 — CONFUSION MATRIX

print("\n" + "=" * 60)
print("EXPERIMENT 8: Confusion Matrix — WESAD")
print("=" * 60)

# Train a fresh HABC on the ablation split (same split for consistency)
K.clear_session()
model_cm = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_cm.fit([X_cnn_tr_ab, X_lstm_tr_ab], y_tr_ab,
             epochs=100, batch_size=64, verbose=0, callbacks=[es])

y_pred_cm = np.argmax(model_cm.predict([X_cnn_te_ab, X_lstm_te_ab], verbose=0), axis=1)
y_true_cm = y_te_ab_raw

# ── Raw counts confusion matrix ──
cm = confusion_matrix(y_true_cm, y_pred_cm)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
            linewidths=0.5, linecolor='lightgray')
ax.set(xlabel='Predicted', ylabel='True',
       title='Confusion Matrix — HABC on WESAD (Cross-Subject)')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# ── Normalized confusion matrix ──
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
            linewidths=0.5, linecolor='lightgray', vmin=0, vmax=1)
ax.set(xlabel='Predicted', ylabel='True',
       title='Normalized Confusion Matrix — HABC on WESAD')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix_norm.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved → exp8_confusion_matrix.png + exp8_confusion_matrix_norm.png")


In [ ]:
# CELL 10: EXPERIMENT 10 — CLASS-WISE METRICS

print("\n" + "=" * 60)
print("EXPERIMENT 10: Class-Wise Precision, Recall, F1 — WESAD")
print("=" * 60)

report = classification_report(y_true_cm, y_pred_cm,
                               target_names=CLASS_LIST,
                               output_dict=True, zero_division=0)

# Print table
print(f"\n{'='*65}")
print(f"  {'Class':<15} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print(f"{'='*65}")

for cls_name in CLASS_LIST:
    r = report[cls_name]
    print(f"  {cls_name:<15} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
          f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")

print(f"  {'-'*60}")
for avg in ['macro avg', 'weighted avg']:
    r = report[avg]
    label = avg.title()
    print(f"  {label:<15} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
          f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")
print(f"{'='*65}")

overall = accuracy_score(y_true_cm, y_pred_cm)
print(f"\n  Overall Accuracy: {overall*100:.2f}%")

# Full standard report
print(f"\n--- Full Classification Report ---")
print(classification_report(y_true_cm, y_pred_cm,
                            target_names=CLASS_LIST, digits=4, zero_division=0))

# Per-class F1 bar chart
fig, ax = plt.subplots(figsize=(8, 5))
f1_vals = [report[c]['f1-score']*100 for c in CLASS_LIST]
colors = ['#4CAF50' if v >= 80 else '#FF9800' if v >= 50 else '#F44336' for v in f1_vals]
ax.bar(CLASS_LIST, f1_vals, color=colors, edgecolor='gray', linewidth=0.5)
ax.axhline(np.mean(f1_vals), color='blue', ls='--', alpha=0.7,
           label=f'Mean F1 = {np.mean(f1_vals):.1f}%')
ax.set(ylabel='F1-Score (%)', title='Per-Class F1-Score — HABC on WESAD')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp10_per_class_f1.png", dpi=300, bbox_inches='tight')
plt.show()

exp10 = {
    "experiment": "Exp10_ClassWise", "dataset": "WESAD",
    "overall_accuracy": float(overall),
    "report": report
}
with open(f"{RESULTS_DIR}/exp10_classwise.json", "w") as f:
    json.dump(exp10, f, indent=2)
print(f"Saved → exp10_classwise.json + exp10_per_class_f1.png")



In [ ]:
# CELL 11: FINAL SUMMARY
###############################################################################

print("\n" + "=" * 70)
print("  FINAL SUMMARY — ALL WESAD EXPERIMENTS COMPLETE")
print("=" * 70)

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│  Dataset: WESAD │ Samples: {len(raw_X):,} │ Classes: {NUM_CLASSES} │ Subjects: {len(np.unique(raw_subjects))}         │
├─────────────────────────────────────────────────────────────────────┤
│  Exp 1: Cross-Subject Generalization                               │
│    LOSO Accuracy:      {np.mean(loso_accs)*100:6.2f}% ± {np.std(loso_accs)*100:.2f}%                        │
│    LOSO Macro-F1:      {np.mean(loso_mf1)*100:6.2f}% ± {np.std(loso_mf1)*100:.2f}%                        │
│    Cross-Sub Accuracy: {np.mean(cs_accs)*100:6.2f}% ± {np.std(cs_accs)*100:.2f}%                        │
├─────────────────────────────────────────────────────────────────────┤
│  Exp 2: Ablation Study                                             │""")
for n, r in ablation_results.items():
    tag = " ◀" if "HABC" in n else "  "
    print(f"│    {n:<28} Acc={r['accuracy']*100:5.2f}%  F1={r['macro_f1']*100:5.2f}% {tag} │")
print(f"""├─────────────────────────────────────────────────────────────────────┤
│  Exp 3: Robustness                                                 │
│    Clean: {clean_acc*100:.2f}%  SNR=10dB: {noise_results.get('snr_10',0)*100:.2f}%  Missing 20%: {miss_results.get('20%',0)*100:.2f}%   │
├─────────────────────────────────────────────────────────────────────┤
│  Exp 4: Statistical Significance                                   │""")
for bn, sr in sig_results.items():
    mark = "✓ significant" if sr['significant_005'] else "✗ not sig."
    print(f"│    HABC vs {bn:<18} p={sr['p_ttest']:.4f} → {mark:<16}│")
print(f"""├─────────────────────────────────────────────────────────────────────┤
│  Exp 8:  Confusion matrices saved ✓                                │
│  Exp 10: Class-wise metrics saved ✓                                │
├─────────────────────────────────────────────────────────────────────┤
│  All files saved to: {RESULTS_DIR}/            │
└─────────────────────────────────────────────────────────────────────┘
""")

# Master summary JSON
master = {
    "model": "HABC (CNN + BiLSTM + Attention)",
    "dataset": "WESAD",
    "num_samples": int(len(raw_X)),
    "num_classes": NUM_CLASSES,
    "num_subjects": int(len(np.unique(raw_subjects))),
    "experiments": [
        "Exp1: Cross-Subject (LOSO + Cross-Subject Split)",
        "Exp2: Ablation Study (5 variants)",
        "Exp3: Robustness (Gaussian Noise + Missing Data)",
        "Exp4: Statistical Significance (t-test + Wilcoxon)",
        "Exp8: Confusion Matrix (raw + normalized)",
        "Exp10: Class-Wise Metrics + Per-Class F1 chart",
    ],
    "output_files": [
        "sanity_training_curves.png",
        "exp2_ablation.png",
        "exp3_robustness.png",
        "exp8_confusion_matrix.png",
        "exp8_confusion_matrix_norm.png",
        "exp10_per_class_f1.png",
        "exp1_cross_subject.json",
        "exp2_ablation.json",
        "exp3_robustness.json",
        "exp4_significance.json",
        "exp10_classwise.json",
        "master_summary.json",
    ]
}
with open(f"{RESULTS_DIR}/master_summary.json", "w") as f:
    json.dump(master, f, indent=2)

print("ALL WESAD EXPERIMENTS COMPLETE!")
print(f"Check Google Drive: {RESULTS_DIR}/")